# 第 10 章 · 会话与历史：持久化、恢复、分叉和回退

工具修改文件后进程突然退出，恢复时没有找到成功事件，能否直接重做？不能。
文件可能已经改了，只是结果没提交；结果也可能已经提交，只是回执没送达。
因此必须区分 **已接受 append、已提交 commit、调用方已收到回应**，并把日志、聊天可见历史和工作区分开。

本章复用 SQLite 局部 journal 原型，真实创建数据库并启动两个受控崩溃子进程。
SQLite 事务覆盖日志及其元数据，不覆盖旁边的工作区文件。实验只证明进程终止窗口，不证明断电、坏盘或任意 I/O 故障下的持久性。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
storage = load_example('10', 'journal')
import subprocess
import json
run_dir = fresh_run_dir('ch10')

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. append 已接受，不等于另一个连接已经读到


    append 分配稳定 event ID 并保存脱离调用方的序列化内容。flush 才提交 pending 批次。
    用第二个 SQLite 连接观察提交前后，避免把内存里能看到的内容误称为持久事实。
    本原型选择显式 flush；Codex 的后台 writer 还可能自动写入，因此不能把此时机直接套用到它。


In [3]:
database = run_dir / 'visibility.sqlite'
writer = storage.Journal(database)
reader = storage.Journal(database)
payload = {'text': '修复筛选'}
try:
    event_id = writer.append('user', payload, items=[{'role': 'user', 'content': payload['text']}])
    payload['text'] = '调用方随后修改了字典'
    assert len(writer.pending) == 1
    assert reader.replay() == []
    receipt = writer.flush()
    observed = reader.replay()
    assert observed[0]['event_id'] == event_id
    assert observed[0]['payload']['text'] == '修复筛选'
    assert receipt['committed_seq'] == 1 and writer.pending == []
finally:
    reader.close()
    writer.close()
{'event_id': event_id, 'receipt': receipt, 'other_connection_observed': observed}


{'event_id': '28c778fd-dbde-48a9-9583-39484e5f204c',
 'receipt': {'commit_generation': 1, 'committed_seq': 1},
 'other_connection_observed': [{'event_id': '28c778fd-dbde-48a9-9583-39484e5f204c',
   'items': [{'content': '修复筛选', 'role': 'user'}],
   'kind': 'user',
   'payload': {'text': '修复筛选'}}]}

## 2. 事务失败后，保持同一事件身份再提交


    明确发生在 COMMIT 前的故障可以回滚本批数据库写入；pending 仍由 writer 持有。
    重试不能重新分配事件 ID，否则同一个输入可能变成两条事件。
    这里是显式故障注入，不是已经复现了真实磁盘故障；只有这种已知回滚边界才可以直接按同一批次重试。


In [4]:
journal = storage.Journal(run_dir / 'rollback.sqlite')
try:
    identity = journal.append('tool_intent', {'call_id': 'c1'})
    failure_point = 'before_commit'
    error = expect_error(OSError, lambda: journal.flush(fail_at=failure_point), 'injected')
    assert journal.replay() == [] and len(journal.pending) == 1
    assert journal.pending[0][0] == identity
    journal.flush()
    replayed = journal.replay()
    assert len(replayed) == 1 and replayed[0]['event_id'] == identity
    assert journal.unknown_side_effects() == [{'call_id': 'c1', 'status': 'unknown'}]
finally:
    journal.close()
{'injected_failure': error, 'same_identity_after_retry': replayed[0]['event_id'] == identity}


{'injected_failure': 'injected pre-COMMIT failure',
 'same_identity_after_retry': True}

## 3. 真实终止两个子进程，比较恢复事实


    A 在修改文件后、提交结果前退出；B 在提交结果后、返回前退出。两者都没有普通成功回执，
    文件也都是 after，但恢复日志不同。A 必须报告 unknown，B 能读到已经提交的 tool_result。
    恢复只读取事实，绝不调用文件工具“补做”。修改退出点必须同时修改预期退出码，不能把任意非零退出当成预期故障。


In [5]:
crash_script = ROOT / 'code/mini_codex/components/sessions/crash_writer.py'
crash_reports = []
for mode, expected_code in [('after_file', 73), ('after_commit', 74)]:
    case_dir = run_dir / mode
    completed = subprocess.run([sys.executable, str(crash_script), mode, str(case_dir)],
        capture_output=True, text=True, timeout=10,
        env={'PATH': '/usr/bin:/bin', 'LANG': 'C'})
    (run_dir / (mode + '.stdout.txt')).write_text(completed.stdout)
    (run_dir / (mode + '.stderr.txt')).write_text(completed.stderr)
    assert completed.returncode == expected_code, (completed.returncode, completed.stderr)
    recovered = storage.Journal(case_dir / 'journal.sqlite')
    try:
        events = recovered.replay()
        unknown = recovered.unknown_side_effects()
        assert (case_dir / 'task.txt').read_text() == 'after'
        if mode == 'after_file':
            assert unknown == [{'call_id': 'patch-1', 'status': 'unknown'}]
            assert [e['kind'] for e in events] == ['tool_intent']
        else:
            assert unknown == []
            assert [e['kind'] for e in events] == ['tool_intent', 'tool_result']
        crash_reports.append({'mode': mode, 'actual_exit_code': completed.returncode,
                              'event_kinds': [e['kind'] for e in events], 'unknown': unknown})
    finally:
        recovered.close()
crash_reports


[{'mode': 'after_file',
  'actual_exit_code': 73,
  'event_kinds': ['tool_intent'],
  'unknown': [{'call_id': 'patch-1', 'status': 'unknown'}]},
 {'mode': 'after_commit',
  'actual_exit_code': 74,
  'event_kinds': ['tool_intent', 'tool_result'],
  'unknown': []}]

## 4. 分叉冻结历史前缀，回退不撤回文件


    分叉记录父分支当时的可见事件，不是一个持续追随父分支的引用。回退只改变聊天可见列表，
    不能把磁盘文件恢复到旧内容。提交水位记录全局已提交事实，不随聊天回退倒拨。
    `fork_request_id` 也是幂等身份：同一参数重试返回原回执，复用 ID 改参数则冲突。


In [6]:
journal = storage.Journal(run_dir / 'timeline.sqlite')
workspace_file = run_dir / 'workspace-note.txt'
workspace_file.write_text('真实工作区当前内容')
try:
    for text in ['任务', '第一次观察']:
        journal.append('message', {'text': text}, items=[{'content': text}])
    journal.flush()
    parent_revision = journal.head('main')[0]
    fork_request_id = 'fork-once'
    fork_receipt = journal.timeline(fork_request_id, 'fork', 'main', parent_revision, child='branch-b')
    assert journal.timeline(fork_request_id, 'fork', 'main', parent_revision, child='branch-b') == fork_receipt
    journal.append('message', {'text': '父分支后续观察'})
    journal.flush()
    assert len(journal.replay('main')) == 3 and len(journal.replay('branch-b')) == 2
    before_watermark = journal.watermark()
    journal.timeline('revert-once', 'revert', 'main', journal.head('main')[0], cutoff=1)
    assert len(journal.replay('main')) == 1 and len(journal.replay('branch-b')) == 2
    assert journal.watermark()['committed_seq'] == before_watermark['committed_seq']
    assert journal.watermark()['commit_generation'] > before_watermark['commit_generation']
    assert workspace_file.read_text() == '真实工作区当前内容'
    conflict = expect_error(storage.Conflict,
        lambda: journal.timeline(fork_request_id, 'fork', 'main', parent_revision, child='different-child'), 'reused')
    view_sizes = {b: len(journal.replay(b)) for b in ('main', 'branch-b')}
finally:
    journal.close()
{'view_sizes': view_sizes, 'fork_receipt': fork_receipt, 'changed_request_rejected': conflict}


{'view_sizes': {'main': 1, 'branch-b': 2},
 'fork_receipt': {'request_id': 'fork-once',
  'action': 'fork',
  'branch': 'branch-b',
  'revision': 0},
 'changed_request_rejected': 'request ID reused for different operation'}

## 5. 同一分支的旧 writer 不能覆盖新 head


    两个 writer 都从同一 head 准备数据，先提交的一方会推进 revision。第二方必须看到 stale 冲突，
    不能把新 head 覆盖成自己的旧快照。本原型依靠 SQLite 序列化事务和 head 比较；它不是完整跨主机 writer lease 实现。


In [7]:
db = run_dir / 'writers.sqlite'
first_writer, stale_writer = storage.Journal(db), storage.Journal(db)
try:
    first_id = first_writer.append('message', {'writer': 'first'})
    stale_writer.append('message', {'writer': 'stale'})
    first_writer.flush()
    conflict = expect_error(storage.Conflict, stale_writer.flush, 'stale head')
    assert len(stale_writer.pending) == 1
    assert [e['event_id'] for e in first_writer.replay()] == [first_id]
finally:
    first_writer.close()
    stale_writer.close()
{'stale_commit': conflict, 'committed_event': first_id}


{'stale_commit': 'stale head: retry requires explicit reconciliation',
 'committed_event': '03109abd-94c3-4566-91f6-a9da75262d7f'}

## 6. 损坏数据不能默默变成恢复成功


    日志是恢复依据，校验失败时必须报告不可信的位置或原因。本局部实现采用严格 checksum 拒绝；
    Codex 的兼容读取路径对部分格式错误会计数并跳过，不能把我们的策略当成它的全部行为。
    这里故意改动本实验自己的 SQLite checksum，不触碰任何真实会话文件。


In [8]:
journal = storage.Journal(run_dir / 'corruption.sqlite')
try:
    journal.append('message', {'text': '原始事实'})
    journal.flush()
    journal.db.execute("UPDATE events SET checksum='injected-invalid-checksum'")
    corruption = expect_error(ValueError, journal.replay, 'checksum mismatch')
finally:
    journal.close()
{'recovery_error': corruption, 'artifacts': str(run_dir)}


{'recovery_error': 'canonical event checksum mismatch',
 'artifacts': '<project>/runs/notebooks/ch10-20261113T124117Z-c31b08c2'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [RolloutRecorder::{record_canonical_items,persist,flush}](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/rollout/src/recorder.rs#L1087-L1140)：record_canonical_items 发送 AddItems 后返回，未等待该批文件写入回执；persist/flush 各自排队并等待 oneshot 确认。

- [RolloutWriterState::{write_pending_with_recovery,enter_recovery_mode,write_pending_items_once}](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/rollout/src/recorder.rs#L1789-L1903)：首次写入失败后丢弃句柄重开再试一次，第二次失败返回错误；已成功写出的项从 pending 前缀移除。

- [JsonlWriter::write_line](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/rollout/src/recorder.rs#L2070-L2075)：具体 JSONL 写入路径调用 write_all 和 flush，没有在此调用 sync_all/sync_data。


## 现在应能解释什么


    能区分日志提交、调用回执、模型历史和工作区现状；恢复时先读取事实，把未闭合副作用标为 unknown，
    再由具体工具检查现场。分叉与回退改变可见时间线，不自动撤销文件或网络操作。

    接回原 Loop 时替换持久化适配器，并由调度器负责停止 producer、flush 与 writer 交接。
    本章已经提供可执行的真实 SQLite/进程崩溃实验，但是否通过应以本次逐格运行结果为准；
    不涵盖断电、磁盘坏块、跨主机租约、完整历史迁移或累计 Agent 检查点。
